# Comprehensive EDA of Brazilian Human Rights Hotline Complaints

## Step 9: Hypothesis Generation and Confirmation Plan

This notebook converts robust findings from Steps 4 through 8 into explicit, testable hypotheses. It separates exploratory discovery from confirmation, defines the analytical unit, estimand, denominator, variables, statistical test, effect size, assumptions, multiplicity control, sensitivity analyses, and decision rule for each hypothesis.

### Objectives

1. Import or register findings from earlier EDA steps.
2. Score findings for evidential strength and confirmability.
3. Translate selected findings into structured hypotheses.
4. Define confirmatory samples without reusing unrestricted discovery data.
5. Generate analysis-ready datasets and statistical test templates.
6. Apply effect-size reporting and false-discovery-rate control.
7. Create sensitivity analyses for counting unit, missingness, sparse categories, time, and geography.
8. Produce a preregistration-style hypothesis register and decision log.

> Hypotheses generated from the same dataset remain exploratory until evaluated in a genuinely independent sample, a future time window, a held-out geographic set, or a pre-specified holdout not used during discovery.


## 9.1 Imports and configuration

The notebook reads the cleaned dataset from Step 3 and, when available, the findings registers from Steps 4 through 8. It does not require every earlier output to exist.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu, kruskal
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 220)

PARQUET_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.parquet")
CSV_PATH = Path("outputs/step_03_data_cleaning/disque100_clean.csv")
OUTPUT_DIR = Path("outputs/step_09_hypothesis_generation")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

FINDING_FILES = {
    "univariate": Path("outputs/step_04_univariate_analysis/tables/univariate_findings_register.csv"),
    "bivariate": Path("outputs/step_05_bivariate_analysis/tables/bivariate_findings_register.csv"),
    "temporal": Path("outputs/step_06_temporal_analysis/tables/temporal_findings.csv"),
    "geographic": Path("outputs/step_07_geographic_analysis/tables/geographic_findings.csv"),
    "multivariate": Path("outputs/step_08_multivariate_exploration/tables/multivariate_findings.csv"),
}

COUNTING_MODE = "record"  # record or complaint
IDENTIFIER = "hash"
DATE_COLUMN = "data_de_cadastro"
ALPHA = 0.05
FDR_METHOD = "benjamini-hochberg"
MIN_EXPECTED = 5
MIN_CELL_PUBLICATION = 5
RANDOM_STATE = 42
CONFIRMATION_STRATEGY = "temporal_holdout"  # temporal_holdout, random_holdout, geographic_holdout, external
DISCOVERY_FRACTION = 0.70
HOLDOUT_UFS = []  # Example: ["DF", "GO"] for geographic_holdout
EXPORT_FIGURES_HTML = True
PLOTLY_TEMPLATE = "plotly_white"


## 9.2 Load the cleaned analytical dataset

The counting unit must match the estimand. Record-level hypotheses describe complaint-linked records, while complaint-level hypotheses describe unique `hash` values. Never switch units after viewing confirmatory results.


In [2]:
def load_clean_dataset():
    if PARQUET_PATH.exists():
        return pd.read_parquet(PARQUET_PATH), PARQUET_PATH
    if CSV_PATH.exists():
        return pd.read_csv(CSV_PATH, low_memory=False), CSV_PATH
    raise FileNotFoundError("Run Step 3 or update PARQUET_PATH/CSV_PATH.")


df, DATA_SOURCE = load_clean_dataset()
if DATE_COLUMN in df.columns:
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
if COUNTING_MODE not in {"record", "complaint"}:
    raise ValueError("COUNTING_MODE must be 'record' or 'complaint'.")
if COUNTING_MODE == "complaint":
    if IDENTIFIER not in df.columns:
        raise ValueError("Complaint mode requires hash.")
    df_analysis = df.dropna(subset=[IDENTIFIER]).sort_values(DATE_COLUMN).drop_duplicates(IDENTIFIER).copy()
    unit_label = "complaint"
else:
    df_analysis = df.copy()
    unit_label = "record"

print("Source:", DATA_SOURCE.resolve())
print(f"Analytical population: {len(df_analysis):,} {unit_label}s")


Source: C:\Projetos\eda-disque100-completo\eda\outputs\step_03_data_cleaning\disque100_clean.parquet
Analytical population: 8,401 records


## 9.3 Consolidate findings from previous EDA steps

Earlier findings are treated as leads, not confirmed results. The source step is retained so that every hypothesis has an audit trail back to the exploratory evidence.


In [3]:
finding_frames=[]
for source,path in FINDING_FILES.items():
    if not path.exists():
        continue
    table=pd.read_csv(path)
    table["source_step"]=source
    table["source_file"]=str(path)
    finding_frames.append(table)
all_findings=pd.concat(finding_frames,ignore_index=True,sort=False) if finding_frames else pd.DataFrame()
print(f"Imported {len(all_findings):,} findings from {len(finding_frames)} prior steps.")
display(all_findings.head(100))


Imported 56 findings from 4 prior steps.


,column,finding_type,finding,implication,source_step,source_file,pair,interpretation,scope
0,deficiencia_do_suspeito,high_missingness,77.13% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.,univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
1,deficiencia_da_vitima,high_missingness,76.72% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.,univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
2,motivacao,high_missingness,65.05% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.,univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
3,raca_cor_do_suspeito,high_missingness,55.61% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.,univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
4,suspeito_preso,high_missingness,51.95% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.,univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
5,faixa_etaria_do_suspeito,high_missingness,45.64% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.,univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
6,raca_cor_da_vitima,high_missingness,40.71% missing,Report the denominator and evaluate missingness mechanisms before subgroup analysis.,univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
7,municipio,high_cardinality,313 valid categories,"Use top-N charts and review grouping, hierarchy, spelling, and disclosure risk.",univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
8,denuncia_emergencial,dominant_category,Top category represents 91.01% of valid observations,"Verify whether this reflects the phenomenon, a default response, or collection design.",univariate,outputs\step_04_univariate_analysis\tables\univariate_findings_register.csv,NaN,NaN,NaN
9,NaN,association,FDR p=1.109e-169; Cramér's V=0.102 (small),NaN,bivariate,outputs\step_05_bivariate_analysis\tables\bivariate_findings_register.csv,violacao × faixa_etaria_da_vitima,Describe the association and inspect standardized residuals; avoid causal language.,NaN


## 9.4 Evidence-triage framework

Each exploratory finding is assessed against six dimensions:

- analytical-unit clarity
- denominator clarity
- data completeness
- category stability and cell size
- effect-size evidence
- independent confirmation feasibility

The score prioritizes hypotheses for confirmation. It is not a probability that a hypothesis is true.


In [4]:
EVIDENCE_CRITERIA = [
    "analytical_unit_clear",
    "denominator_clear",
    "completeness_acceptable",
    "sparse_cells_controlled",
    "effect_size_available",
    "independent_confirmation_feasible",
]

# Fill manually after reviewing the imported findings.
evidence_triage = pd.DataFrame(columns=[
    "candidate_id", "source_step", "source_reference", "candidate_statement",
    *EVIDENCE_CRITERIA, "priority_notes"
])

# Example structure only. Replace with findings from this research.
EXAMPLE_CANDIDATES = [
    {
        "candidate_id":"C01",
        "source_step":"bivariate",
        "source_reference":"violacao × faixa_etaria_da_vitima",
        "candidate_statement":"The distribution of recorded violation categories differs across victim age groups.",
        "analytical_unit_clear":1,
        "denominator_clear":1,
        "completeness_acceptable":1,
        "sparse_cells_controlled":0,
        "effect_size_available":1,
        "independent_confirmation_feasible":1,
        "priority_notes":"Confirm using a temporal holdout and collapsed official violation hierarchy."
    },
    {
        "candidate_id":"C02",
        "source_step":"bivariate",
        "source_reference":"violacao × relacao_vitima_suspeito",
        "candidate_statement":"Recorded violation composition differs by victim-suspect relationship.",
        "analytical_unit_clear":1,
        "denominator_clear":1,
        "completeness_acceptable":1,
        "sparse_cells_controlled":0,
        "effect_size_available":1,
        "independent_confirmation_feasible":1,
        "priority_notes":"Use FDR control and inspect standardized residuals only after omnibus confirmation."
    },
    {
        "candidate_id":"C03",
        "source_step":"temporal",
        "source_reference":"monthly emergency rate",
        "candidate_statement":"The proportion of complaints classified as emergency changes over calendar time.",
        "analytical_unit_clear":1,
        "denominator_clear":1,
        "completeness_acceptable":1,
        "sparse_cells_controlled":1,
        "effect_size_available":0,
        "independent_confirmation_feasible":1,
        "priority_notes":"Pre-specify periods and account for form or channel changes."
    },
]

if evidence_triage.empty:
    evidence_triage=pd.DataFrame(EXAMPLE_CANDIDATES)
evidence_triage["evidence_score"] = evidence_triage[EVIDENCE_CRITERIA].sum(axis=1)
evidence_triage["priority"] = pd.cut(
    evidence_triage["evidence_score"], bins=[-1,2,4,6], labels=["low","medium","high"]
)
display(evidence_triage.sort_values("evidence_score",ascending=False))


,candidate_id,source_step,source_reference,candidate_statement,analytical_unit_clear,denominator_clear,completeness_acceptable,sparse_cells_controlled,effect_size_available,independent_confirmation_feasible,priority_notes,evidence_score,priority
0,C01,bivariate,violacao × faixa_etaria_da_vitima,The distribution of recorded violation categories differs across victim age groups.,1,1,1,0,1,1,Confirm using a temporal holdout and collapsed official violation hierarchy.,5,high
1,C02,bivariate,violacao × relacao_vitima_suspeito,Recorded violation composition differs by victim-suspect relationship.,1,1,1,0,1,1,Use FDR control and inspect standardized residuals only after omnibus confirmation.,5,high
2,C03,temporal,monthly emergency rate,The proportion of complaints classified as emergency changes over calendar time.,1,1,1,1,0,1,Pre-specify periods and account for form or channel changes.,5,high


## 9.5 Structured hypothesis register

Every hypothesis must specify:

- null and alternative hypotheses
- analytical unit and eligible population
- outcome and exposure
- estimand and denominator
- expected direction, if genuinely pre-specified
- primary statistical test and effect size
- multiplicity family
- missing-data and sparse-category strategy
- confirmation sample
- sensitivity analyses
- decision rule

Edit this register before running confirmatory tests.


In [5]:
HYPOTHESES = [
    {
        "hypothesis_id":"H01",
        "status":"proposed",
        "research_question":"Does recorded violation composition differ across victim age groups?",
        "null_hypothesis":"Violation category is independent of victim age group among eligible recorded observations.",
        "alternative_hypothesis":"Violation category is associated with victim age group among eligible recorded observations.",
        "analytical_unit":COUNTING_MODE,
        "eligible_population":"Observations with non-missing violation and victim age group in the confirmatory sample.",
        "exposure":"faixa_etaria_da_vitima",
        "outcome":"violacao",
        "estimand":"Bias-corrected Cramér's V for the age-group by violation contingency table.",
        "denominator":"Complete observations for both variables after pre-specified category collapsing.",
        "direction":"two-sided",
        "primary_test":"Pearson chi-square test of independence",
        "effect_size":"Bias-corrected Cramér's V",
        "multiplicity_family":"primary_categorical",
        "missing_strategy":"complete pair; report missingness by age group",
        "sparse_strategy":"use official hierarchy; require no expected cell below 1 and <=20% below 5",
        "confirmation_sample":CONFIRMATION_STRATEGY,
        "sensitivity_analyses":"record vs complaint; exclude partial date periods; alternate rare-category threshold",
        "decision_rule":"FDR-adjusted p < 0.05, valid expected-cell diagnostics, and non-negligible effect size",
    },
    {
        "hypothesis_id":"H02",
        "status":"proposed",
        "research_question":"Does recorded violation composition differ by victim-suspect relationship?",
        "null_hypothesis":"Violation category is independent of victim-suspect relationship.",
        "alternative_hypothesis":"Violation category is associated with victim-suspect relationship.",
        "analytical_unit":COUNTING_MODE,
        "eligible_population":"Observations with non-missing violation and relationship values in the confirmatory sample.",
        "exposure":"relacao_vitima_suspeito",
        "outcome":"violacao",
        "estimand":"Bias-corrected Cramér's V for relationship by violation.",
        "denominator":"Complete observations for both variables after pre-specified category collapsing.",
        "direction":"two-sided",
        "primary_test":"Pearson chi-square test of independence",
        "effect_size":"Bias-corrected Cramér's V",
        "multiplicity_family":"primary_categorical",
        "missing_strategy":"complete pair; compare missingness across violation groups",
        "sparse_strategy":"collapse only through documented relationship and violation hierarchies",
        "confirmation_sample":CONFIRMATION_STRATEGY,
        "sensitivity_analyses":"alternative hierarchy; exclude missing-intensive periods; stratify by vulnerable group",
        "decision_rule":"FDR-adjusted p < 0.05, assumptions acceptable, and effect size reported",
    },
    {
        "hypothesis_id":"H03",
        "status":"proposed",
        "research_question":"Is emergency classification associated with recorded violation category?",
        "null_hypothesis":"Emergency classification is independent of violation category.",
        "alternative_hypothesis":"Emergency classification is associated with violation category.",
        "analytical_unit":COUNTING_MODE,
        "eligible_population":"Observations with valid emergency and violation categories in the confirmatory sample.",
        "exposure":"violacao",
        "outcome":"denuncia_emergencial",
        "estimand":"Difference in emergency proportions across violation categories and Cramér's V.",
        "denominator":"Valid emergency and violation observations.",
        "direction":"two-sided",
        "primary_test":"Pearson chi-square test of independence",
        "effect_size":"Bias-corrected Cramér's V and category-specific emergency proportion",
        "multiplicity_family":"primary_categorical",
        "missing_strategy":"complete pair; report unknown emergency classification separately",
        "sparse_strategy":"collapse violation categories using a pre-specified official hierarchy",
        "confirmation_sample":CONFIRMATION_STRATEGY,
        "sensitivity_analyses":"channel-stratified analysis; time-stratified analysis; complaint-count sensitivity",
        "decision_rule":"FDR-adjusted p < 0.05 with acceptable assumptions and stable direction across sensitivity analyses",
    },
    {
        "hypothesis_id":"H04",
        "status":"proposed",
        "research_question":"Does the number of victims per record differ across vulnerable groups?",
        "null_hypothesis":"The distribution of victim quantity is equal across vulnerable groups.",
        "alternative_hypothesis":"At least one vulnerable group has a different victim-quantity distribution.",
        "analytical_unit":COUNTING_MODE,
        "eligible_population":"Observations with valid vulnerable group and positive victim quantity in the confirmatory sample.",
        "exposure":"grupo_vulneravel",
        "outcome":"sl_quantidade_vitimas",
        "estimand":"Difference in distribution and group medians of victim quantity.",
        "denominator":"Eligible observations with both fields present.",
        "direction":"two-sided",
        "primary_test":"Kruskal-Wallis test",
        "effect_size":"Epsilon-squared and group medians with IQR",
        "multiplicity_family":"primary_numeric",
        "missing_strategy":"complete pair; report missingness by vulnerable group",
        "sparse_strategy":"groups below the pre-specified minimum are combined only through a documented hierarchy",
        "confirmation_sample":CONFIRMATION_STRATEGY,
        "sensitivity_analyses":"winsorization-free; exclude collective complaints above pre-specified percentile; record vs complaint",
        "decision_rule":"FDR-adjusted p < 0.05 and practically meaningful median differences",
    },
]

hypothesis_register=pd.DataFrame(HYPOTHESES)
display(hypothesis_register)


,hypothesis_id,status,research_question,null_hypothesis,alternative_hypothesis,analytical_unit,eligible_population,exposure,outcome,estimand,denominator,direction,primary_test,effect_size,multiplicity_family,missing_strategy,sparse_strategy,confirmation_sample,sensitivity_analyses,decision_rule
0,H01,proposed,Does recorded violation composition differ across victim age groups?,Violation category is independent of victim age group among eligible recorded observations.,Violation category is associated with victim age group among eligible recorded observations.,record,Observations with non-missing violation and victim age group in the confirmatory sample.,faixa_etaria_da_vitima,violacao,Bias-corrected Cramér's V for the age-group by violation contingency table.,Complete observations for both variables after pre-specified category collapsing.,two-sided,Pearson chi-square test of independence,Bias-corrected Cramér's V,primary_categorical,complete pair; report missingness by age group,use official hierarchy; require no expected cell below 1 and <=20% below 5,temporal_holdout,record vs complaint; exclude partial date periods; alternate rare-category threshold,"FDR-adjusted p < 0.05, valid expected-cell diagnostics, and non-negligible effect size"
1,H02,proposed,Does recorded violation composition differ by victim-suspect relationship?,Violation category is independent of victim-suspect relationship.,Violation category is associated with victim-suspect relationship.,record,Observations with non-missing violation and relationship values in the confirmatory sample.,relacao_vitima_suspeito,violacao,Bias-corrected Cramér's V for relationship by violation.,Complete observations for both variables after pre-specified category collapsing.,two-sided,Pearson chi-square test of independence,Bias-corrected Cramér's V,primary_categorical,complete pair; compare missingness across violation groups,collapse only through documented relationship and violation hierarchies,temporal_holdout,alternative hierarchy; exclude missing-intensive periods; stratify by vulnerable group,"FDR-adjusted p < 0.05, assumptions acceptable, and effect size reported"
2,H03,proposed,Is emergency classification associated with recorded violation category?,Emergency classification is independent of violation category.,Emergency classification is associated with violation category.,record,Observations with valid emergency and violation categories in the confirmatory sample.,violacao,denuncia_emergencial,Difference in emergency proportions across violation categories and Cramér's V.,Valid emergency and violation observations.,two-sided,Pearson chi-square test of independence,Bias-corrected Cramér's V and category-specific emergency proportion,primary_categorical,complete pair; report unknown emergency classification separately,collapse violation categories using a pre-specified official hierarchy,temporal_holdout,channel-stratified analysis; time-stratified analysis; complaint-count sensitivity,FDR-adjusted p < 0.05 with acceptable assumptions and stable direction across sensitivity analyses
3,H04,proposed,Does the number of victims per record differ across vulnerable groups?,The distribution of victim quantity is equal across vulnerable groups.,At least one vulnerable group has a different victim-quantity distribution.,record,Observations with valid vulnerable group and positive victim quantity in the confirmatory sample.,grupo_vulneravel,sl_quantidade_vitimas,Difference in distribution and group medians of victim quantity.,Eligible observations with both fields present.,two-sided,Kruskal-Wallis test,Epsilon-squared and group medians with IQR,primary_numeric,complete pair; report missingness by vulnerable group,groups below the pre-specified minimum are combined only through a documented hierarchy,temporal_holdout,winsorization-free; exclude collective complaints above pre-specified percentile; record vs complaint,FDR-adjusted p < 0.05 and practically meaningful median differe

## 9.6 Validate the hypothesis register

The validator checks completeness, variable availability, duplicate identifiers, and whether hypotheses were explicitly marked as proposed or confirmatory.


In [6]:
REQUIRED_HYPOTHESIS_FIELDS = [
    "hypothesis_id","status","research_question","null_hypothesis","alternative_hypothesis",
    "analytical_unit","eligible_population","exposure","outcome","estimand","denominator",
    "primary_test","effect_size","multiplicity_family","missing_strategy","sparse_strategy",
    "confirmation_sample","sensitivity_analyses","decision_rule"
]

validation_rows=[]
for _,h in hypothesis_register.iterrows():
    missing_fields=[c for c in REQUIRED_HYPOTHESIS_FIELDS if c not in h.index or pd.isna(h[c]) or str(h[c]).strip()==""]
    absent_variables=[v for v in [h["exposure"],h["outcome"]] if v not in df_analysis.columns]
    validation_rows.append({
        "hypothesis_id":h["hypothesis_id"],
        "duplicate_id":int((hypothesis_register["hypothesis_id"]==h["hypothesis_id"]).sum())>1,
        "missing_required_fields":"; ".join(missing_fields),
        "absent_variables":"; ".join(absent_variables),
        "status_valid":h["status"] in {"proposed","preregistered","tested","retired"},
        "ready":not missing_fields and not absent_variables and h["status"] in {"proposed","preregistered"}
    })
hypothesis_validation=pd.DataFrame(validation_rows)
display(hypothesis_validation)


,hypothesis_id,duplicate_id,missing_required_fields,absent_variables,status_valid,ready
0,H01,False,,,True,True
1,H02,False,,,True,True
2,H03,False,,,True,True
3,H04,False,,,True,True


## 9.7 Create discovery and confirmation samples

### Available strategies

- `temporal_holdout`: earliest 70% of dated observations for discovery, latest 30% for confirmation
- `random_holdout`: stratified only if explicitly added; otherwise reproducible random split
- `geographic_holdout`: specified UFs are reserved for confirmation
- `external`: no internal confirmation sample is created

A temporal holdout is usually preferable for administrative systems because it tests whether a pattern persists into a later period, but it can be affected by form or policy changes.


In [7]:
def create_confirmation_split(data,strategy):
    result=data.copy()
    result["analysis_partition"]="unassigned"
    if strategy=="temporal_holdout":
        if DATE_COLUMN not in result.columns:
            raise ValueError("Temporal holdout requires the date column.")
        dated=result[result[DATE_COLUMN].notna()].sort_values(DATE_COLUMN)
        cutoff_position=max(1,int(len(dated)*DISCOVERY_FRACTION))
        cutoff_date=dated.iloc[min(cutoff_position,len(dated)-1)][DATE_COLUMN]
        result.loc[result[DATE_COLUMN]<cutoff_date,"analysis_partition"]="discovery"
        result.loc[result[DATE_COLUMN]>=cutoff_date,"analysis_partition"]="confirmation"
        result.loc[result[DATE_COLUMN].isna(),"analysis_partition"]="excluded_missing_date"
        metadata={"strategy":strategy,"cutoff_date":str(cutoff_date)}
    elif strategy=="random_holdout":
        rng=np.random.default_rng(RANDOM_STATE)
        mask=rng.random(len(result))<DISCOVERY_FRACTION
        result.loc[mask,"analysis_partition"]="discovery"
        result.loc[~mask,"analysis_partition"]="confirmation"
        metadata={"strategy":strategy,"random_state":RANDOM_STATE,"discovery_fraction":DISCOVERY_FRACTION}
    elif strategy=="geographic_holdout":
        if "uf" not in result.columns or not HOLDOUT_UFS:
            raise ValueError("Geographic holdout requires uf and non-empty HOLDOUT_UFS.")
        result.loc[~result["uf"].isin(HOLDOUT_UFS),"analysis_partition"]="discovery"
        result.loc[result["uf"].isin(HOLDOUT_UFS),"analysis_partition"]="confirmation"
        metadata={"strategy":strategy,"holdout_ufs":HOLDOUT_UFS}
    elif strategy=="external":
        result["analysis_partition"]="discovery_only"
        metadata={"strategy":strategy,"confirmation":"Provide an independent external dataset."}
    else:
        raise ValueError("Unknown confirmation strategy.")
    return result,metadata


partitioned_data,partition_metadata=create_confirmation_split(df_analysis,CONFIRMATION_STRATEGY)
partition_summary=partitioned_data["analysis_partition"].value_counts(dropna=False).rename_axis("partition").reset_index(name="n")
partition_summary["percent"]=(partition_summary["n"]/len(partitioned_data)*100).round(2)
display(partition_summary)
print(partition_metadata)


,partition,n,percent
0,discovery,5880,69.99
1,confirmation,2521,30.01


{'strategy': 'temporal_holdout', 'cutoff_date': '2025-04-25 15:44:09'}


## 9.8 Check comparability of discovery and confirmation samples

A holdout is useful only if its differences are understood. The notebook compares missingness and categorical composition. Temporal drift is expected in a temporal holdout and should be reported rather than hidden.


In [8]:
comparison_variables=sorted(set(
    hypothesis_register["exposure"].tolist()+hypothesis_register["outcome"].tolist()
))
partition_quality_rows=[]
for variable in [v for v in comparison_variables if v in partitioned_data.columns]:
    for partition,g in partitioned_data.groupby("analysis_partition"):
        partition_quality_rows.append({
            "variable":variable,"partition":partition,"n":len(g),
            "missing_n":int(g[variable].isna().sum()),
            "missing_pct":round(g[variable].isna().mean()*100,2),
            "unique_valid":int(g[variable].nunique(dropna=True))
        })
partition_quality=pd.DataFrame(partition_quality_rows)
display(partition_quality)

fig=px.bar(partition_quality,x="variable",y="missing_pct",color="partition",barmode="group",title="Missingness in discovery and confirmation partitions",labels={"missing_pct":"Missing (%)"})
fig.update_layout(template=PLOTLY_TEMPLATE,xaxis_tickangle=-45)
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"partition_missingness.html",include_plotlyjs=True)


,variable,partition,n,missing_n,missing_pct,unique_valid
0,denuncia_emergencial,confirmation,2521,0,0.00,2
1,denuncia_emergencial,discovery,5880,0,0.00,5
2,faixa_etaria_da_vitima,confirmation,2521,4,0.16,27
3,faixa_etaria_da_vitima,discovery,5880,471,8.01,39
4,grupo_vulneravel,confirmation,2521,0,0.00,6
5,grupo_vulneravel,discovery,5880,0,0.00,9
6,relacao_vitima_suspeito,confirmation,2521,32,1.27,16
7,relacao_vitima_suspeito,discovery,5880,285,4.85,40
8,sl_quantidade_vitimas,confirmation,2521,0,0.00,2
9,sl_quantidade_vitimas,discovery,5880,0,0.00,10


## 9.9 Statistical helper functions

The functions return effect sizes, assumption diagnostics, and complete-case denominators. They do not automatically collapse domains because category mappings must be pre-specified in the hypothesis register.


In [9]:
def cramers_v_bias_corrected(table):
    chi2=chi2_contingency(table,correction=False)[0]
    n=table.to_numpy().sum()
    r,k=table.shape
    if n<=1: return np.nan
    phi2=chi2/n
    phi2corr=max(0,phi2-((k-1)*(r-1))/(n-1))
    rcorr=r-((r-1)**2)/(n-1)
    kcorr=k-((k-1)**2)/(n-1)
    denominator=min(kcorr-1,rcorr-1)
    return np.sqrt(phi2corr/denominator) if denominator>0 else np.nan


def categorical_test(data,exposure,outcome):
    pair=data[[exposure,outcome]].dropna()
    table=pd.crosstab(pair[exposure],pair[outcome])
    if table.shape[0]<2 or table.shape[1]<2:
        return {"status":"insufficient_variation","n_complete":len(pair)},table,pd.DataFrame()
    chi2,p,dof,expected=chi2_contingency(table,correction=False)
    expected_df=pd.DataFrame(expected,index=table.index,columns=table.columns)
    residuals=(table-expected_df)/np.sqrt(expected_df)
    low5=int((expected<MIN_EXPECTED).sum())
    low1=int((expected<1).sum())
    result={
        "status":"completed","n_complete":int(table.to_numpy().sum()),"chi2":chi2,
        "dof":dof,"p_value":p,"cramers_v":cramers_v_bias_corrected(table),
        "expected_below_5":low5,"expected_below_5_pct":round(low5/expected.size*100,2),
        "expected_below_1":low1,"assumptions_ok":low1==0 and low5/expected.size<=.20
    }
    return result,table,residuals


def epsilon_squared_kruskal(statistic,n,k):
    return max(0,(statistic-k+1)/(n-k)) if n>k else np.nan


def numeric_group_test(data,group,outcome):
    pair=data[[group,outcome]].dropna().copy()
    pair[outcome]=pd.to_numeric(pair[outcome],errors="coerce")
    pair=pair.dropna()
    arrays=[g[outcome].to_numpy() for _,g in pair.groupby(group) if len(g)>0]
    if len(arrays)<2:
        return {"status":"insufficient_variation","n_complete":len(pair)},pd.DataFrame()
    if len(arrays)==2:
        statistic,p=mannwhitneyu(arrays[0],arrays[1],alternative="two-sided")
        effect=np.nan
        test="Mann-Whitney U"
    else:
        statistic,p=kruskal(*arrays)
        effect=epsilon_squared_kruskal(statistic,len(pair),len(arrays))
        test="Kruskal-Wallis"
    summary=pair.groupby(group)[outcome].agg(n="count",median="median",mean="mean",q1=lambda s:s.quantile(.25),q3=lambda s:s.quantile(.75)).reset_index()
    return {"status":"completed","test":test,"n_complete":len(pair),"groups":len(arrays),"statistic":statistic,"p_value":p,"epsilon_squared":effect},summary


def benjamini_hochberg(p_values):
    p=np.asarray(p_values,dtype=float)
    n=len(p)
    order=np.argsort(p)
    ranked=p[order]
    adjusted=ranked*n/np.arange(1,n+1)
    adjusted=np.minimum.accumulate(adjusted[::-1])[::-1]
    result=np.empty(n)
    result[order]=np.clip(adjusted,0,1)
    return result


## 9.10 Lock the hypothesis register before confirmation

Set `RUN_CONFIRMATORY_TESTS = True` only after the register, category mappings, holdout strategy, multiplicity families, and decision rules have been frozen. The lock file records the exact specification used.


In [10]:
RUN_CONFIRMATORY_TESTS = False

lock_payload = {
    "hypotheses": HYPOTHESES,
    "counting_mode": COUNTING_MODE,
    "confirmation_strategy": CONFIRMATION_STRATEGY,
    "partition_metadata": partition_metadata,
    "alpha": ALPHA,
    "fdr_method": FDR_METHOD,
    "minimum_expected": MIN_EXPECTED,
    "data_source": str(DATA_SOURCE),
}
lock_text=json.dumps(lock_payload,ensure_ascii=False,sort_keys=True,indent=2)
import hashlib
specification_hash=hashlib.sha256(lock_text.encode("utf-8")).hexdigest()
(OUTPUT_DIR/"hypothesis_specification_locked.json").write_text(lock_text,encoding="utf-8")
print("Specification SHA-256:",specification_hash)
print("Confirmatory tests enabled:",RUN_CONFIRMATORY_TESTS)


Specification SHA-256: ef8de8917747b047fe6c9e9d5291fe43ed1d6900493db835f13d2f9042e34bca
Confirmatory tests enabled: False


## 9.11 Execute confirmatory tests

Results are generated only from the confirmation partition. If `RUN_CONFIRMATORY_TESTS` is false, the notebook produces preparation artifacts without revealing confirmatory p-values.


In [11]:
confirmatory_results=[]
residual_outputs={}
summary_outputs={}

if RUN_CONFIRMATORY_TESTS:
    confirmation=partitioned_data[partitioned_data["analysis_partition"]=="confirmation"].copy()
    if confirmation.empty:
        raise ValueError("Confirmation sample is empty.")
    for _,h in hypothesis_register[hypothesis_register["status"].isin(["proposed","preregistered"])].iterrows():
        hid=h["hypothesis_id"]
        exposure,outcome=h["exposure"],h["outcome"]
        if h["primary_test"]=="Pearson chi-square test of independence":
            result,table,residuals=categorical_test(confirmation,exposure,outcome)
            result.update({"hypothesis_id":hid,"family":h["multiplicity_family"],"exposure":exposure,"outcome":outcome})
            confirmatory_results.append(result)
            summary_outputs[hid]=table
            residual_outputs[hid]=residuals
        elif h["primary_test"]=="Kruskal-Wallis test":
            result,summary=numeric_group_test(confirmation,exposure,outcome)
            result.update({"hypothesis_id":hid,"family":h["multiplicity_family"],"exposure":exposure,"outcome":outcome})
            confirmatory_results.append(result)
            summary_outputs[hid]=summary

confirmatory_results=pd.DataFrame(confirmatory_results)
if not confirmatory_results.empty:
    complete=confirmatory_results[confirmatory_results["status"]=="completed"].copy()
    adjusted_parts=[]
    for family,g in complete.groupby("family"):
        g=g.copy()
        g["p_fdr_bh"]=benjamini_hochberg(g["p_value"])
        g["significant_fdr"]=g["p_fdr_bh"]<ALPHA
        adjusted_parts.append(g)
    adjusted=pd.concat(adjusted_parts,ignore_index=True) if adjusted_parts else pd.DataFrame()
    confirmatory_results=confirmatory_results.drop(columns=[c for c in ["p_fdr_bh","significant_fdr"] if c in confirmatory_results]).merge(
        adjusted[["hypothesis_id","p_fdr_bh","significant_fdr"]],on="hypothesis_id",how="left"
    )
display(confirmatory_results)


""


## 9.12 Decision framework

A hypothesis is not classified from the p-value alone. The decision incorporates multiplicity control, assumptions, effect size, direction, and sensitivity consistency.


In [12]:
decision_rows=[]
if not confirmatory_results.empty:
    for _,r in confirmatory_results.iterrows():
        h=hypothesis_register[hypothesis_register["hypothesis_id"]==r["hypothesis_id"]].iloc[0]
        assumptions_ok=bool(r.get("assumptions_ok",True))
        significant=bool(r.get("significant_fdr",False))
        if r["status"]!="completed":
            decision="not_testable"
        elif not assumptions_ok:
            decision="inconclusive_assumption_failure"
        elif significant:
            decision="statistically_supported_pending_sensitivity"
        else:
            decision="not_supported_in_confirmation_sample"
        decision_rows.append({
            "hypothesis_id":r["hypothesis_id"],"decision":decision,
            "fdr_adjusted_significant":significant,"assumptions_ok":assumptions_ok,
            "effect_size":r.get("cramers_v",r.get("epsilon_squared",np.nan)),
            "decision_rule":h["decision_rule"],
            "final_interpretation":"Complete sensitivity analyses before final classification."
        })
hypothesis_decisions=pd.DataFrame(decision_rows)
display(hypothesis_decisions)


""


## 9.13 Sensitivity-analysis matrix

The matrix is a protocol, not an invitation to select the most favorable result. All pre-specified variants should be reported together.


In [13]:
SENSITIVITY_PLAN = [
    {"analysis_id":"S01","dimension":"counting unit","variant":"record-level","purpose":"Assess repeated-record influence."},
    {"analysis_id":"S02","dimension":"counting unit","variant":"unique hash","purpose":"Assess complaint-level robustness."},
    {"analysis_id":"S03","dimension":"missingness","variant":"complete pair","purpose":"Primary pairwise analysis."},
    {"analysis_id":"S04","dimension":"missingness","variant":"missing as category where substantively valid","purpose":"Assess documentation-pattern influence."},
    {"analysis_id":"S05","dimension":"time","variant":"exclude partial boundary months","purpose":"Avoid incomplete-period distortion."},
    {"analysis_id":"S06","dimension":"time","variant":"exclude documented form/system transition periods","purpose":"Assess collection-process shifts."},
    {"analysis_id":"S07","dimension":"category hierarchy","variant":"official detailed categories","purpose":"Primary domain representation."},
    {"analysis_id":"S08","dimension":"category hierarchy","variant":"official collapsed hierarchy","purpose":"Improve sparse-cell diagnostics."},
    {"analysis_id":"S09","dimension":"geography","variant":"exclude invalid or unmatched geographic records","purpose":"Assess geographic-quality influence."},
    {"analysis_id":"S10","dimension":"outliers","variant":"retain all positive victim quantities","purpose":"Primary numeric analysis."},
    {"analysis_id":"S11","dimension":"outliers","variant":"exclude pre-specified extreme collective complaints","purpose":"Assess tail sensitivity without silent deletion."},
]
sensitivity_plan=pd.DataFrame(SENSITIVITY_PLAN)
display(sensitivity_plan)


,analysis_id,dimension,variant,purpose
0,S01,counting unit,record-level,Assess repeated-record influence.
1,S02,counting unit,unique hash,Assess complaint-level robustness.
2,S03,missingness,complete pair,Primary pairwise analysis.
3,S04,missingness,missing as category where substantively valid,Assess documentation-pattern influence.
4,S05,time,exclude partial boundary months,Avoid incomplete-period distortion.
5,S06,time,exclude documented form/system transition periods,Assess collection-process shifts.
6,S07,category hierarchy,official detailed categories,Primary domain representation.
7,S08,category hierarchy,official collapsed hierarchy,Improve sparse-cell diagnostics.
8,S09,geography,exclude invalid or unmatched geographic records,Assess geographic-quality influence.
9,S10,outliers,retain all positive victim quantities,Primary numeric analysis.


## 9.14 Power and minimum detectable effect planning

Administrative datasets often make very small effects statistically significant. The research should define a smallest effect size of interest before confirmation. The table below is a planning register; formal power calculations should use the final table dimensions, allocation, alpha after multiplicity, and expected sample size.


In [14]:
power_plan=pd.DataFrame([
    {
        "hypothesis_family":"primary_categorical",
        "effect_metric":"Cramér's V",
        "smallest_effect_of_interest":0.10,
        "alpha":ALPHA,
        "target_power":0.80,
        "planning_notes":"Recalculate using final contingency-table degrees of freedom and multiplicity-adjusted alpha."
    },
    {
        "hypothesis_family":"primary_numeric",
        "effect_metric":"epsilon-squared or median difference",
        "smallest_effect_of_interest":"Define with domain experts",
        "alpha":ALPHA,
        "target_power":0.80,
        "planning_notes":"Use simulation based on discovery-sample group sizes and empirical distributions."
    }
])
display(power_plan)


,hypothesis_family,effect_metric,smallest_effect_of_interest,alpha,target_power,planning_notes
0,primary_categorical,Cramér's V,0.1,0.05,0.8,Recalculate using final contingency-table degrees of freedom and multiplicity-adjusted alpha.
1,primary_numeric,epsilon-squared or median difference,Define with domain experts,0.05,0.8,Use simulation based on discovery-sample group sizes and empirical distributions.


## 9.15 Hypothesis traceability matrix

The traceability table connects the exploratory finding, hypothesis, variables, confirmatory sample, outputs, and final decision. This prevents undocumented movement from discovery to testing.


In [15]:
traceability=hypothesis_register[[
    "hypothesis_id","research_question","exposure","outcome","estimand",
    "multiplicity_family","confirmation_sample","decision_rule"
]].copy()
traceability=traceability.merge(
    evidence_triage[["candidate_id","source_step","source_reference","candidate_statement","evidence_score"]],
    left_index=True,right_index=True,how="left"
)
if not hypothesis_decisions.empty:
    traceability=traceability.merge(hypothesis_decisions[["hypothesis_id","decision"]],on="hypothesis_id",how="left")
display(traceability)


,hypothesis_id,research_question,exposure,outcome,estimand,multiplicity_family,confirmation_sample,decision_rule,candidate_id,source_step,source_reference,candidate_statement,evidence_score
0,H01,Does recorded violation composition differ across victim age groups?,faixa_etaria_da_vitima,violacao,Bias-corrected Cramér's V for the age-group by violation contingency table.,primary_categorical,temporal_holdout,"FDR-adjusted p < 0.05, valid expected-cell diagnostics, and non-negligible effect size",C01,bivariate,violacao × faixa_etaria_da_vitima,The distribution of recorded violation categories differs across victim age groups.,5.0
1,H02,Does recorded violation composition differ by victim-suspect relationship?,relacao_vitima_suspeito,violacao,Bias-corrected Cramér's V for relationship by violation.,primary_categorical,temporal_holdout,"FDR-adjusted p < 0.05, assumptions acceptable, and effect size reported",C02,bivariate,violacao × relacao_vitima_suspeito,Recorded violation composition differs by victim-suspect relationship.,5.0
2,H03,Is emergency classification associated with recorded violation category?,violacao,denuncia_emergencial,Difference in emergency proportions across violation categories and Cramér's V.,primary_categorical,temporal_holdout,FDR-adjusted p < 0.05 with acceptable assumptions and stable direction across sensitivity analyses,C03,temporal,monthly emergency rate,The proportion of complaints classified as emergency changes over calendar time.,5.0
3,H04,Does the number of victims per record differ across vulnerable groups?,grupo_vulneravel,sl_quantidade_vitimas,Difference in distribution and group medians of victim quantity.,primary_numeric,temporal_holdout,FDR-adjusted p < 0.05 and practically meaningful median differences,NaN,NaN,NaN,NaN,NaN


## 9.16 Interactive hypothesis dashboard

The dashboard shows hypothesis readiness and evidence-triage scores. It does not display confirmatory p-values unless testing has been explicitly enabled.


In [16]:
dashboard=evidence_triage.copy()
fig=px.bar(
    dashboard.sort_values("evidence_score"),x="evidence_score",y="candidate_id",
    orientation="h",color="priority",hover_data=["candidate_statement","source_step","priority_notes"],
    title="Candidate hypothesis evidence-triage score",
    labels={"evidence_score":"Criteria satisfied","candidate_id":"Candidate"}
)
fig.update_layout(template=PLOTLY_TEMPLATE,xaxis=dict(dtick=1,range=[0,len(EVIDENCE_CRITERIA)]))
fig.show()
if EXPORT_FIGURES_HTML: fig.write_html(FIGURE_DIR/"hypothesis_evidence_triage.html",include_plotlyjs=True)


## 9.17 Export protocol, partitions, and results

The partition assignment and `hash` are restricted analytical outputs. Public outputs should contain aggregate hypothesis specifications and suppressed results only.


In [17]:
exports={
    "consolidated_exploratory_findings.csv":all_findings,
    "evidence_triage.csv":evidence_triage,
    "hypothesis_register.csv":hypothesis_register,
    "hypothesis_validation.csv":hypothesis_validation,
    "partition_summary.csv":partition_summary,
    "partition_quality.csv":partition_quality,
    "confirmatory_results.csv":confirmatory_results,
    "hypothesis_decisions.csv":hypothesis_decisions,
    "sensitivity_plan.csv":sensitivity_plan,
    "power_plan.csv":power_plan,
    "hypothesis_traceability.csv":traceability,
}
for filename,table in exports.items():
    table.to_csv(TABLE_DIR/filename,index=False,encoding="utf-8-sig")

restricted_columns=[c for c in [IDENTIFIER,"analysis_partition"] if c in partitioned_data.columns]
restricted_partition=partitioned_data[restricted_columns].copy()
restricted_partition.insert(0,"source_index",partitioned_data.index)
restricted_partition["public_release_allowed"]=False
restricted_partition.to_csv(TABLE_DIR/"RESTRICTED_partition_assignments.csv",index=False,encoding="utf-8-sig")

if RUN_CONFIRMATORY_TESTS:
    for hid,table in summary_outputs.items():
        table.to_csv(TABLE_DIR/f"{hid}_confirmatory_summary.csv",encoding="utf-8-sig")
    for hid,table in residual_outputs.items():
        table.to_csv(TABLE_DIR/f"{hid}_standardized_residuals.csv",encoding="utf-8-sig")

figure_index=pd.DataFrame({"figure_file":sorted(p.name for p in FIGURE_DIR.glob("*.html"))})
figure_index.to_csv(OUTPUT_DIR/"figure_index.csv",index=False,encoding="utf-8-sig")

metadata={
    "data_source":str(DATA_SOURCE),"counting_mode":COUNTING_MODE,
    "analytical_unit":unit_label,"analytical_rows":int(len(df_analysis)),
    "confirmation_strategy":CONFIRMATION_STRATEGY,
    "partition_metadata":partition_metadata,"alpha":ALPHA,
    "multiplicity_correction":FDR_METHOD,"specification_sha256":specification_hash,
    "confirmatory_tests_run":RUN_CONFIRMATORY_TESTS,
}
(OUTPUT_DIR/"run_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print("Tables:",TABLE_DIR.resolve())
print("Figures:",FIGURE_DIR.resolve())


Tables: C:\Projetos\eda-disque100-completo\eda\outputs\step_09_hypothesis_generation\tables
Figures: C:\Projetos\eda-disque100-completo\eda\outputs\step_09_hypothesis_generation\figures


## 9.18 Interpretation and preregistration checklist

Before changing `RUN_CONFIRMATORY_TESTS` to `True`:

1. Freeze the research questions and hypothesis identifiers.
2. Confirm the analytical unit and denominator.
3. Define the eligible population and exclusions.
4. Freeze category mappings and hierarchy levels.
5. Define the holdout or external confirmation sample.
6. Set the primary test and effect size.
7. Assign each hypothesis to a multiplicity family.
8. Define the smallest effect size of substantive interest.
9. Freeze missingness and sparse-cell rules.
10. Freeze all sensitivity analyses.
11. Record the specification SHA-256.
12. Do not inspect confirmation outcomes before locking the protocol.

### Interpretation categories

- **Supported in confirmation sample:** the pre-specified decision rule is met, assumptions are acceptable, effect size is meaningful, and sensitivity results are coherent.
- **Not supported:** the pre-specified rule is not met in the confirmation sample.
- **Inconclusive:** assumptions fail, precision is insufficient, the holdout is unsuitable, or data quality prevents a valid test.
- **Exploratory only:** the hypothesis was generated and tested on the same observations without independent confirmation.

### Recommended reporting language

“Hypotheses were generated from exploratory analyses and specified before evaluation in a held-out sample. Confirmatory tests used pre-defined analytical units, denominators, category rules, effect sizes, and Benjamini-Hochberg false-discovery-rate control within hypothesis families. Results describe associations among recorded hotline observations and do not establish causality or population incidence.”

## Next step

**Step 10: Communication and Reproducible Reporting** should assemble the validated findings, methods, figures, limitations, correction logs, and disclosure controls into an executive summary, technical report, and publication-ready results package.
